# Tutorial 1 — Why Flow Development Length Matters
**Microfluidic engineering | Dr. Francisco Tovar**

**Goal:** Understand the physical problem, estimate $L_{FD}$ using a published correlation, and explore flow rate, viscosity and Reynolds number. **No CFD in this tutorial.**

## 1. Start with the engineering problem

One of the first challenges in microfluidics is **interfacing the external world with the small world of the microfluidic device**.

**syringe → tubing → connector → inlet port → microchannel**

Changes in size also occur inside the device:

**channel → contraction → sensing region → expansion → another channel**

At each transition the cross-sectional area may change. For incompressible flow,

$$Q=UA,\qquad U=\frac{Q}{A}$$

where $Q$ is flow rate, $U$ is **cross-sectional mean velocity**, and $A$ is area. At fixed $Q$, smaller area means higher mean velocity.

```text
External world                          Microfluidic world
    tubing            connector             microchannel
=============>         ====>        |---------------------->
  large area                          small area
  lower mean U                        higher mean U
```

But the velocity **profile** does not instantly adjust to the new geometry.

> **After a change in channel size or geometry, how much distance is required before the flow stops changing significantly?**

This motivates the **flow development length**, $L_{FD}$.

## 2. What develops?

The no-slip condition makes velocity zero at stationary walls. Viscous effects reshape the velocity profile downstream until its **shape** changes negligibly with distance.

```text
INLET             DEVELOPING              DEVELOPED
 |→|                 | → |                  |  →  |
 |→|                 | →→→ |                | →→→ |
 |→|                 |→→→→→|                |→→→→→|
 |→|                 | →→→ |                | →→→ |
```

**Important:** In a constant-area channel with constant $Q$, the **mean velocity** $U=Q/A$ is already fixed. It is the **velocity distribution** that develops. $L_{FD}$ depends on the criterion used for “sufficiently developed.”

## 3. Why should an engineer care?

**Too short:** A sensor or cell region may experience changing shear rate, wall shear stress, velocity distribution and trajectories. A fully developed-flow assumption could be wrong.

**Unnecessarily long:** More footprint, hydraulic resistance, pressure drop, internal volume, fabrication area and pumping requirements. Biological samples may experience longer residence and contact with PDMS/plastic surfaces, with possible adsorption or pre-exposure effects.

**Design question:** How much entrance length is justified for the intended experiment? Not every device needs fully developed flow.

## 4. How can we estimate $L_{FD}$?

There are several approaches. **This tutorial uses a published correlation only.** A later tutorial will introduce CFD as another approximate model, not as ground truth.

Li, Huang & Wu (2019) report, for **square microchannels**,

$$\boxed{\frac{L_{FD}}{D_h}=\frac{0.28}{1+0.1Re}+0.0537Re}$$

Their study covers **$0.5\le Re\le50$** and hydraulic diameters of 100–200 µm; the correlation fitted their data within approximately 15.2%. We will use it to teach the calculation, including an illustrative **rectangular** channel, but **its accuracy for that rectangle is not established**.

**Assumptions and risks:** The calculation uses a representative mean velocity and a homogeneous, Newtonian, incompressible, laminar fluid with constant properties. Mean velocity is **not** a uniform velocity profile. Geometry, inlet conditions and the criterion for development may differ from the published study. A correlation is an **estimate under assumptions**, not a measurement.

**Reference:** Li, H., Huang, B., & Wu, M. (2019), *Experimental and Numerical Investigations on the Flow Characteristics within Hydrodynamic Entrance Regions in Microchannels*, **Micromachines 10**, 317. DOI: 10.3390/mi10050317.

## 5. What information do we need?

For a rectangular channel of width $w$ and height $h$:

$$A=wh,\qquad D_h=\frac{2wh}{w+h}.$$

Then

$$U=\frac{Q}{A},\qquad Re=\frac{\rho U D_h}{\mu}.$$

We must specify **flow rate $Q$, width $w$, height $h$, density $\rho$, and dynamic viscosity $\mu$**. Convert to SI units before calculating.

## 6. Worked example: calculate the length step by step

Assume $w=200\ \mu\mathrm{m}$, $h=85\ \mu\mathrm{m}$, $Q=100\ \mu\mathrm{L/min}$, $\rho=1000\ \mathrm{kg/m^3}$ and $\mu=1\ \mathrm{mPa\cdot s}$.

**1. Area:**
$$A=(200\times10^{-6})(85\times10^{-6})=1.70\times10^{-8}\ \mathrm{m^2}.$$

**2. Hydraulic diameter:**
$$D_h=\frac{2(200)(85)}{200+85}=119.3\ \mu\mathrm{m}.$$

**3. Convert flow rate:**
$$Q=\frac{100\times10^{-9}}{60}=1.667\times10^{-9}\ \mathrm{m^3/s}.$$

**4. Mean velocity:**
$$U=\frac{1.667\times10^{-9}}{1.70\times10^{-8}}=0.0980\ \mathrm{m/s}.$$

**5. Reynolds number:**
$$Re=\frac{(1000)(0.0980)(119.3\times10^{-6})}{0.001}=11.7.$$

**6. Correlation:**
$$\frac{L_{FD}}{D_h}=\frac{0.28}{1+0.1(11.7)}+0.0537(11.7)\approx0.757.$$

**7. Estimated length:**
$$L_{FD}\approx0.757(119.3\ \mu\mathrm{m})\approx90\ \mu\mathrm{m}=0.090\ \mathrm{mm}.$$

**Interpretation:** A correlation-based teaching estimate, **not a validated inlet length** for this rectangular geometry.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Five editable inputs
w_um, h_um = 200.0, 85.0
Q_uL_min, rho, mu_mPa_s = 100.0, 1000.0, 1.0
w, h = w_um*1e-6, h_um*1e-6
A = w*h
Dh = 2*w*h/(w+h)
Q = Q_uL_min*1e-9/60
mu = mu_mPa_s*1e-3
U = Q/A
Re = rho*U*Dh/mu

def correlation(Re):
    """Li et al. 2019, Eq. 7: tested for square microchannels, Re 0.5–50."""
    return 0.28/(1+0.1*np.asarray(Re)) + 0.0537*np.asarray(Re)

LFD = correlation(Re)*Dh
print(f"Area: {A:.3e} m² | Dh: {Dh*1e6:.1f} µm")
print(f"Mean U: {U:.4f} m/s | Re: {Re:.2f}")
print(f"Estimated LFD: {LFD*1e3:.4f} mm ({LFD*1e6:.1f} µm)")

## 7. Explore flow rates and viscosities

Now keep geometry and density fixed. Vary $Q$ and $\mu$, and use **the same published correlation** for each case.

**Predict first:** What happens to $Re$ when $Q$ increases? What happens when viscosity increases? What does that imply for estimated $L_{FD}$?

In [ ]:
flow_rates = np.array([10,25,50,100,200,400])  # µL/min
viscosities = np.array([1,2,4,8])          # mPa·s
rows=[]
for viscosity in viscosities:
    for flow in flow_rates:
        q = flow*1e-9/60
        u = q/A
        re = rho*u*Dh/(viscosity*1e-3)
        rows.append({"Q [µL/min]":flow,"μ [mPa·s]":viscosity,"U [m/s]":u,
                     "Re":re,"LFD [µm]":float(correlation(re)*Dh*1e3),
                     "LFD/Dh":float(correlation(re)),"Within Re 0.5–50":0.5<=re<=50})
df=pd.DataFrame(rows)
display(df.round(4))
print("Within published Re range:",df["Within Re 0.5–50"].sum(),"of",len(df))

In [ ]:
fig,ax=plt.subplots(figsize=(8,5))
for viscosity,g in df.groupby("μ [mPa·s]"):
    ax.plot(g["Q [µL/min]"],g["LFD [µm]"],marker="o",label=f"μ={viscosity:g} mPa·s")
ax.set(xlabel="Flow rate Q [µL/min]",ylabel="Estimated LFD [µm]",title="Effect of flow rate and viscosity")
ax.grid(alpha=.3);ax.legend();plt.tight_layout();plt.show()

## 8. Explore different Reynolds-number regimes

The published study reports different **entrance-length trends** at lower $Re$ and higher $Re$, with a change around $Re\approx12.5$–15. This is **not** the transition from laminar to turbulent flow.

Explore the correlation across **its published range**, $0.5\le Re\le50$. What happens as $Re$ increases? Is the relationship exactly linear?

In [ ]:
re_axis=np.linspace(.5,50,300)
fig,ax=plt.subplots(figsize=(8,5))
ax.plot(re_axis,correlation(re_axis),label="Li et al. (2019), Eq. 7")
ax.axvspan(12.5,15,alpha=.12,label="Reported change in trend")
ax.set(xlabel="Reynolds number Re [-]",ylabel="LFD/Dh [-]",title="Low and higher Reynolds numbers")
ax.grid(alpha=.3);ax.legend();plt.tight_layout();plt.show()

## 9. Do the different curves collapse?

Reynolds number combines $Q$, $\mu$, $\rho$ and channel geometry:

$$Re=\frac{\rho QD_h}{\mu A}.$$

Plot $L_{FD}/D_h$ versus $Re$ for every flow rate and viscosity. **This collapse is expected by construction**, because all points came from a correlation that already uses $Re$ as its input. It is **not independent evidence** that real experiments will collapse.

In [ ]:
fig,ax=plt.subplots(figsize=(8,5))
ax.plot(re_axis,correlation(re_axis),alpha=.45,label="Published correlation")
for viscosity,g in df.groupby("μ [mPa·s]"):
    valid=g[g["Within Re 0.5–50"]]
    ax.scatter(valid["Re"],valid["LFD/Dh"],label=f"μ={viscosity:g} mPa·s")
ax.set(xlabel="Reynolds number Re [-]",ylabel="LFD/Dh [-]",title="Organising the estimates by Reynolds number")
ax.grid(alpha=.3);ax.legend();plt.tight_layout();plt.show()

## 10. What does this mean for the device?

A correlation gives an **initial engineering estimate**, not a guaranteed minimum length. Before choosing the device length, ask:

1. Which operating condition gives the largest estimated $L_{FD}$?
2. Are all the predictions within the correlation's published $Re$ range?
3. Does the channel geometry match the study (square versus rectangular)?
4. What extra footprint, residence time and pressure drop would a longer entrance section create?
5. Does the experiment actually require fully developed flow?

The ideal mean-flow residence time over a length $L$ is $t=L/U$. This is an average transit-time estimate, not the residence time of every particle.

In [ ]:
# Compare the residence time of a candidate length and one five times longer
candidate_mm = float(df["LFD [µm]"].max())
for factor in (1,5):
    length_m = factor*candidate_mm*1e-3
    t_ms = 1e3*length_m/U  # mean velocity from the worked example
    print(f"Length = {factor} × max predicted LFD = {factor*candidate_mm:.3f} mm; "
          f"mean transit time at worked-example Q = {t_ms:.2f} ms")
print("These values are illustrative; no safety factor or validity guarantee is implied.")

## 11. Your engineering exercise

1. Change the flow-rate and viscosity arrays. Which condition has the largest $L_{FD}$ estimate?
2. Find two combinations of $Q$ and $\mu$ that produce similar $Re$. Do they produce similar estimated $L_{FD}$?
3. Change the channel height. What happens to $U$, $D_h$, $Re$ and $L_{FD}$?
4. Identify any cases outside the published $Re$ range. Why should they be treated cautiously?
5. Explain why using $U=Q/A$ does **not** assume uniform velocity everywhere.
6. Explain why an accurate numerical answer can still be an unsuitable engineering design estimate.

### Take-home message

**Start with the physical connection and the engineering need. Calculate with stated assumptions. Explore operating conditions. Then look for simpler relationships.**

**Next tutorial (not included here):** CFD-based evaluation of velocity-profile development and the assumptions and numerical checks involved.